# **EU Projects**

Authors: Andrea Bincoletto & Enrico Longato @ Area Science Park

Description: Python script to filter and concatenate data on European projects for FVG companies.

Files downloaded from: https://cordis.europa.eu/projects

The Framework Programs to download are:

- Horizon 2020 (folder "cordis-h2020projects-csv") link: https://data.europa.eu/data/datasets/cordish2020projects?locale=en
- Horizon Europe (folder "cordis-HORIZONprojects-csv") link: https://data.europa.eu/data/datasets/cordis-eu-research-projects-under-horizon-europe-2021-2027?locale=en

Each folder contains the following .csv files:

- euroSciVoc
- organization
- project
- legalBasis
- topics
- webItem
- webLink

In [1]:
# Setup
import sys
import os
from pathlib import Path
import datetime
import pandas as pd
import numpy as np
import io
import sharepy
import getpass
import openpyxl
import csv
pd.options.display.max_columns = None

## **Company Registry**

**Load the list of FVG companies**

In [ ]:
base_path = Path.cwd().parent   
file_anagrafica = base_path / "data" / "anagrafica" / "iifvg_anagrafica_filtrato.csv"

print("Using this file:", file_anagrafica)

df_anagrafica = pd.read_csv(file_anagrafica, sep="|", engine="python", dtype=str)

# keep only active companies (only active companies are shown in I2FVG)
# df_anagrafica = df_anagrafica[df_anagrafica['stato_impresa'].isin(['ATTIVA'])]

filter_i2fvg = df_anagrafica[['piva','cf']]
filter_i2fvg = filter_i2fvg.drop_duplicates('piva')

print(f"The dataframe contains {filter_i2fvg.shape[0]} unique VAT numbers.")

Uso questo file: c:\Users\longato\OneDrive - Area Science Park\I2 - Evoluzione\FAIR\data\anagrafica\iifvg_anagrafica_filtrato.csv
Il dataframe è composto da 98129 partite iva univoche.


## **Processing Framework Program CSV Files**

1 - Access the files through Sharepoint authentication.

2 - The first step is to open the CSV file of the I2FVG registry (source Insiel), updated monthly and contained in the Data Repository. Save the VAT numbers and corresponding fiscal codes in a dataframe.

3 - Open the Cordis CSV files (organizations, projects and euroscivoc) contained in the folder and filter the data for Italian cases, keeping information for companies and projects throughout Italy (initially insert the downloaded folders in the "input" folder).

4 - Finally, concatenate the dataframes of Horizon2020 and HorizonEurope.

NOTE: Use VAT numbers and not fiscal codes because Cordis uses those for companies! Corresponding fiscal codes are also inserted because PowerBI uses the fiscal code as a key between queries.

### **H2020**

#### **Organization**

In [3]:

file = base_path / "data" / "eu_projects" / "h2020" / "organization.csv"
df_h2020_organization = pd.read_csv(file, sep=";", encoding = 'utf-8-sig',dtype = str)  # Use the code below if you get the following error:
                                                                                        # "UnicodeDecodeError: 'utf-8' codec can't decode byte 0xbf in position 176070: invalid start byte"
# df_h2020_organization = pd.read_csv(file, sep=";", encoding = 'unicode_escape',dtype = str)

print(f"The dataframe contains {df_h2020_organization.shape[0]} total (non-unique) companies that participated in H2020 European projects.")

The dataframe contains 178756 total (non-unique) companies that participated in H2020 European projects.


In [4]:
# dataframe with a column containing the coordinator reference by projectID (to be inserted in the project csv)
df_h2020_coordinator = df_h2020_organization.copy()

df_h2020_coordinator = df_h2020_coordinator[['projectID', 'name', 'role']]

df_h2020_coordinator = df_h2020_coordinator.loc[df_h2020_coordinator['role'].isin(['coordinator'])]

df_h2020_coordinator = df_h2020_coordinator.groupby('projectID')['name'].apply(';'.join).reset_index()
df_h2020_coordinator = df_h2020_coordinator.rename(columns={
    'name':'coordinator',
    'projectID':'id'
})

In [5]:
# dataframe with a column containing the partner reference per projectID (to be inserted in the project csv)
df_h2020_partners = df_h2020_organization.copy()

df_h2020_partners = df_h2020_partners[['projectID', 'name', 'role']]

# df_h2020_partners = df_h2020_partners.loc[~df_h2020_partners['role'].isin(['coordinator'])]       # consider also 'thirdParty', 'InternationalPartner' and 'partner'
df_h2020_partners = df_h2020_partners.loc[df_h2020_partners['role'].isin(['participant'])]

df_h2020_partners = df_h2020_partners.groupby('projectID')['name'].apply(';'.join).reset_index()
df_h2020_partners = df_h2020_partners.rename(columns={
    'name':'participants',
    'projectID':'id'
})

In [6]:
# keep only italian companies ("country" = IT)
df_h2020_organization = df_h2020_organization[df_h2020_organization['country'].isin(['IT'])]
print(f"Il dataframe contiene {df_h2020_organization.shape[0]} imprese italiane che hanno partecipato a progetti europei.")

Il dataframe contiene 17241 imprese italiane che hanno partecipato a progetti europei.


In [ ]:
# filter out rows with invalid VAT numbers
df_h2020_organization = (
    df_h2020_organization
    .loc[~df_h2020_organization['vatNumber'].isin(
        ['MISSING', 'NOTAPPLICABLE', 'VATEXEMPTION']
    )]
    .copy()
)

# replace empty strings with NaN
df_h2020_organization['vatNumber'] = (
    df_h2020_organization['vatNumber'].replace('', np.nan)
)

# remove NaN values
df_h2020_organization = df_h2020_organization.dropna(subset=['vatNumber'])

In [8]:
# Strip "IT" in vatNumber column
df_h2020_organization['vatNumber'] = df_h2020_organization['vatNumber'].map(lambda vatNumber: str(vatNumber)[2:])

In [ ]:
# drop rows of VAT numbers that do not have 11 digits
df_h2020_organization['vatnumber_lenght'] = df_h2020_organization['vatNumber'].apply(lambda number: len(number))
print("The VAT numbers in the dataframe have the following lengths: ")
print(df_h2020_organization['vatnumber_lenght'].value_counts().head(10))

df_h2020_organization = df_h2020_organization.loc[df_h2020_organization['vatnumber_lenght'].isin([11])]

print('--------')
print(f"After cleaning, the dataframe contains {df_h2020_organization.shape[0]} Italian companies that participated in European projects.")

Le partite iva del dataframe hanno lunghezza: 
vatnumber_lenght
11    16314
10        2
9         1
12        1
Name: count, dtype: int64
--------
Dopo la pulizia, il dataframe contiene 16314 imprese italiane che hanno partecipato a progetti europei.


In [10]:
# replace dot with comma in "ecContribution" and "netEcContribution"
list_column = ['ecContribution','netEcContribution']
for i in list_column:
    df_h2020_organization[i] = df_h2020_organization[i].str.replace('.',',')
    print("Done!")

Done!
Done!


In [11]:
# merge with dataframes with tax codes of i2fvg companies
df_h2020_organization = pd.merge(df_h2020_organization, filter_i2fvg, how='left', left_on='vatNumber', right_on='piva')

In [12]:
# drop vatnumber_lenght column
df_h2020_organization = df_h2020_organization.drop('vatnumber_lenght', axis=1)
df_h2020_organization = df_h2020_organization.drop('piva', axis=1)

In [ ]:
# filter VAT numbers by filter_i2fvg (not to be used): check for FVG
df_h2020_organization_i2fvg = df_h2020_organization[df_h2020_organization['vatNumber'].isin(filter_i2fvg['piva'])]
print(f"The total number of FVG companies with matches in European projects is {df_h2020_organization_i2fvg.shape[0]}.")
print(f"The unique FVG companies in I2FVG that participated in H2020 European projects are {df_h2020_organization_i2fvg['vatNumber'].nunique()}.")

Il totale delle imprese fvg con corrispondenza in progetti europei è 1115.
Le imprese fvg univoche contenute in I2FVG che hanno partecipato a progetti europei H2020 sono 187.


#### **Project**

In [ ]:

file = base_path / "data" / "eu_projects" / "h2020" / "project.csv"



df_h2020_project = pd.read_csv(file, sep=";", encoding="utf-8-sig", dtype=str, quotechar='"', on_bad_lines="skip")
print(f"The dataframe contains {df_h2020_project.shape[0]} European projects.")

Il dataframe contiene 35105 progetti europei.


In [15]:
# Insert columns with reference to the coordinator and project partner
df_h2020_project = pd.merge(left=df_h2020_project, right=df_h2020_coordinator, on='id', how='left')
df_h2020_project = pd.merge(left=df_h2020_project, right=df_h2020_partners, on='id', how='left')

In [ ]:
# Filter European projects of Italian companies only
filter_IT_h2020_projects = df_h2020_organization['projectID']
filter_IT_h2020_projects = filter_IT_h2020_projects.drop_duplicates()

df_h2020_project = df_h2020_project[df_h2020_project['id'].isin(filter_IT_h2020_projects)]
print(f"The total number of European projects is {df_h2020_project.shape[0]}.")

Il totale dei progetti europei è 7588.


In [ ]:
# Filter FVG projects (not to be used)
filter_i2fvg_h2020_project = df_h2020_organization_i2fvg['projectID']
filter_i2fvg_h2020_project = filter_i2fvg_h2020_project.drop_duplicates()
print(f"The total number of European projects from I2FVG companies is {filter_i2fvg_h2020_project.shape[0]}.")

Il totale dei progetti europei delle imprese I2FVG è 870.


#### **euroSciVoc**

In [18]:

file = base_path / "data" / "eu_projects" / "h2020" / "euroSciVoc.csv"


df_h2020_euroSciVoc = pd.read_csv(file, sep=";", encoding="utf-8-sig", dtype=str, quotechar='"', on_bad_lines="skip")
print(f"The dataframe contains {df_h2020_euroSciVoc.shape[0]} euroSciVoc entries.")

The dataframe contains 112111 euroSciVoc entries.


In [ ]:
print(f"The total number of unique euroSciVocTitle entries is {df_h2020_euroSciVoc['euroSciVocTitle'].drop_duplicates().count()}.")
print(f"The main euroSciVocTitle entries are: \n{df_h2020_euroSciVoc['euroSciVocTitle'].value_counts().head(5)}")

Gli euroSciVocTitle univoci sono in totale 1051.
I principali euroSciVocTitle sono: 
euroSciVocTitle
software           1788
sensors            1780
business models    1651
proteins           1504
oncology           1286
Name: count, dtype: int64


In [20]:
# filter euroSciVoc with filter_IT_h2020_projects
df_h2020_euroscivoc = df_h2020_euroSciVoc[df_h2020_euroSciVoc['projectID'].isin(filter_IT_h2020_projects)]
df_h2020_euroscivoc.shape

(23306, 5)

In [54]:
# create a column with the first level of the euroscivoc path (6 in total)
df_h2020_euroscivoc = df_h2020_euroscivoc.copy()

df_h2020_euroscivoc['livello_1'] = (
    df_h2020_euroscivoc['euroSciVocPath']
        .str.split('/')
        .str[1]
)

df_h2020_euroscivoc['livello_1'].value_counts().head(6)

livello_1
natural sciences               7637
engineering and technology     7242
social sciences                4182
medical and health sciences    2835
agricultural sciences           777
humanities                      633
Name: count, dtype: int64

In [ ]:
# create a column with the second level of the euroscivoc path (if missing, the first level is inserted)
df_h2020_euroscivoc = df_h2020_euroscivoc.copy()

df_h2020_euroscivoc['livello_2'] = (
    df_h2020_euroscivoc['euroSciVocPath']
        .str.split('/')
        .str[2]                     # extract level 2 directly
        .fillna(df_h2020_euroscivoc['livello_1'])   # fallback to level 1
)

print(df_h2020_euroscivoc['livello_2'].value_counts().head(5))

livello_2
computer and information sciences                                          2749
electrical engineering, electronic engineering, information engineering    2043
environmental engineering                                                  2042
economics and business                                                     1617
biological sciences                                                        1524
Name: count, dtype: int64


In [ ]:
# filter the euroSciVocTitle and livello_1 for I2FVG companies (not to be used)
filter_i2fvg_h2020_euroscivoc = df_h2020_euroscivoc[df_h2020_euroscivoc['projectID'].isin(filter_i2fvg_h2020_project)]
print(f"The main euroSciVocTitle entries for I2FVG are: \n{filter_i2fvg_h2020_euroscivoc['euroSciVocTitle'].value_counts().head(5)}")
print("----------------------------------------")
print(f"The livello_1 entries for I2FVG are: \n{filter_i2fvg_h2020_euroscivoc['livello_1'].value_counts()}")
print("----------------------------------------")
print(f"The main livello_2 entries for I2FVG are: \n{filter_i2fvg_h2020_euroscivoc['livello_2'].value_counts().head(5)}")

I principali euroSciVocTitle i2FVG sono: 
euroSciVocTitle
business models               70
sensors                       70
software                      60
climatic change mitigation    52
governance                    47
Name: count, dtype: int64
----------------------------------------
I livello_1 i2FVG sono: 
livello_1
engineering and technology     999
natural sciences               860
social sciences                518
medical and health sciences    171
agricultural sciences          109
humanities                      21
Name: count, dtype: int64
----------------------------------------
I principali livello_2 i2FVG sono: 
livello_2
computer and information sciences                                          371
electrical engineering, electronic engineering, information engineering    307
environmental engineering                                                  287
economics and business                                                     193
biological sciences                 

### **Horizon Europe**

#### **Organization**

In [ ]:

file = base_path / "data" / "eu_projects" / "horizon_europe" / "organization.csv"
df_he_organization = pd.read_csv(file, sep=";", encoding='utf-8-sig', dtype = str)
print(f"The dataframe contains {df_he_organization.shape[0]} total (non-unique) companies that participated in Horizon Europe European projects.")

The dataframe contains 121097 total (non-unique) companies that participated in Horizon Europe European projects.


In [25]:
# dataframe with a column containing the coordinator reference by projectID (to be inserted in the project csv)
df_he_coordinator = df_he_organization.copy()

df_he_coordinator = df_he_coordinator[['projectID', 'name', 'role']]

df_he_coordinator = df_he_coordinator.loc[df_he_coordinator['role'].isin(['coordinator'])]

df_he_coordinator = df_he_coordinator.groupby('projectID')['name'].apply(';'.join).reset_index()
df_he_coordinator = df_he_coordinator.rename(columns={
    'name':'coordinator',
    'projectID':'id'
})

In [26]:
# dataframe with a column containing the partner reference per projectID (to be inserted in the project csv)
df_he_partners = df_he_organization.copy()

df_he_partners = df_he_partners[['projectID', 'name', 'role']]

# df_he_partners = df_he_partners.loc[~df_he_partners['role'].isin(['coordinator'])]       # consider also 'thirdParty', 'InternationalPartner' and 'partner'
df_he_partners = df_he_partners.loc[df_he_partners['role'].isin(['participant'])]

df_he_partners = df_he_partners.groupby(['projectID'], as_index = False).agg({'name': lambda x: ';'.join(x.astype(str))})
df_he_partners = df_he_partners.rename(columns={
    'name':'participants',
    'projectID':'id'
})

In [ ]:
# keep only Italian companies ("country" = IT)
df_he_organization = df_he_organization[df_he_organization['country'].isin(['IT'])]
print(f"The dataframe now contains {df_he_organization.shape[0]} Italian companies that participated in European projects.")

Il dataframe contiene ora 11526 imprese italiane che hanno partecipato a progetti europei.


In [ ]:
# remove missing VAT numbers from the vatNumber column:
# - MISSING
# - VATEXEMPTION
# - NOTAPPLICABLE
df_he_organization = df_he_organization.loc[~df_he_organization['vatNumber'].isin(['MISSING','NOTAPPLICABLE','VATEXEMPTION'])]

# drop blank rows in vatNumber columns
df_he_organization['vatNumber'] = (df_he_organization['vatNumber'].replace('', np.nan))
df_he_organization.dropna(subset=['vatNumber'], inplace=True)

In [29]:
# Strip "IT" in vatNumber column
df_he_organization['vatNumber'] = df_he_organization['vatNumber'].map(lambda vatNumber: str(vatNumber)[2:])

In [ ]:
# drop rows of VAT numbers that do not have 11 digits
df_he_organization['vatnumber_lenght'] = df_he_organization['vatNumber'].apply(lambda number: len(number))
print("The VAT numbers in the dataframe have the following lengths: ")
print(df_he_organization['vatnumber_lenght'].value_counts().head(10))

df_he_organization = df_he_organization.loc[df_he_organization['vatnumber_lenght'].isin([11])]

print('--------')
print(f"After cleaning, the dataframe has {df_he_organization.shape[0]} rows.")

Le partite iva del dataframe hanno lunghezza: 
vatnumber_lenght
11    10859
Name: count, dtype: int64
--------
Dopo la pulizia, il dataframe ha 10859 righe.


In [31]:
# replace dot with comma in "ecContribution" and "netEcContribution"
list_column = ['ecContribution','netEcContribution']
for i in list_column:
    df_he_organization[i] = df_he_organization[i].str.replace('.',',')
    print("Done!")

Done!
Done!


In [32]:
# merge with dataframes with tax codes of i2fvg companies
df_he_organization = pd.merge(df_he_organization, filter_i2fvg, how='left', left_on='vatNumber', right_on='piva')

In [33]:
# drop vatnumber_lenght column
df_he_organization = df_he_organization.drop('vatnumber_lenght', axis=1)
df_he_organization = df_he_organization.drop('piva', axis=1)

In [ ]:
# filter VAT numbers by filter_i2fvg (not to be used)
df_he_organization_i2fvg = df_he_organization[df_he_organization['vatNumber'].isin(filter_i2fvg['piva'])]
print(f"The total number of FVG companies with matches in European projects is {df_he_organization_i2fvg.shape[0]}.")
print(f"The unique FVG companies in I2FVG that participated in Horizon Europe projects are {df_he_organization_i2fvg['vatNumber'].nunique()}.")

Il totale delle imprese fvg con corrispondenza in progetti europei è 734.
Le imprese fvg univoche contenute in I2FVG che hanno partecipato a progetti europei HE sono 160.


#### **Project**

In [35]:


file = base_path / "data" / "eu_projects" / "horizon_europe" / "project.csv"



df_he_project = pd.read_csv(file, sep=";", encoding="utf-8-sig", dtype=str, quotechar='"', on_bad_lines="skip")


print(f"The dataframe contains {df_he_project.shape[0]} European projects.")

The dataframe contains 19335 European projects.


In [36]:
# Insert columns with reference to the coordinator and project partner 
df_he_project = pd.merge(left=df_he_project, right=df_he_coordinator, on='id', how='left')
df_he_project = pd.merge(left=df_he_project, right=df_he_partners, on='id', how='left')

In [ ]:
# Filter European projects of Italian companies only
filter_IT_he_projects = df_he_organization['projectID']
filter_IT_he_projects = filter_IT_he_projects.drop_duplicates()

df_he_project = df_he_project[df_he_project['id'].isin(filter_IT_he_projects)]
print(f"The total number of European projects is {df_he_project.shape[0]}.")

Il totale dei progetti europei è 5255.


In [ ]:
# Filter FVG projects (not to be used)
filter_i2fvg_he_project = df_he_organization_i2fvg['projectID']
filter_i2fvg_he_project = filter_i2fvg_he_project.drop_duplicates()
print(f"The total number of European projects from I2FVG companies is {filter_i2fvg_he_project.shape[0]}.")

Il totale dei progetti europei delle imprese I2FVG è 615.


#### **EuroSciVoc**

In [39]:
file = base_path / "data" / "eu_projects" / "horizon_europe" / "euroSciVoc.csv"

df_he_euroSciVoc = pd.read_csv(file, sep=";", encoding="utf-8-sig", dtype=str, quotechar='"', on_bad_lines="skip")
print(f"The dataframe contains {df_he_euroSciVoc.shape[0]} euroSciVoc entries.")

The dataframe contains 37701 euroSciVoc entries.


In [ ]:
print(f"The total number of unique euroSciVocTitle entries is {df_he_euroSciVoc['euroSciVocTitle'].drop_duplicates().count()}.")
print(f"The main euroSciVocTitle entries are: \n{df_he_euroSciVoc['euroSciVocTitle'].value_counts().head(5)}")

Gli euroSciVocTitle univoci sono in totale 953.
I principali euroSciVocTitle sono: 
euroSciVocTitle
oncology      989
proteins      809
history       758
sensors       659
ecosystems    651
Name: count, dtype: int64


In [41]:
# filter euroSciVoc with filter_IT_projects
df_he_euroscivoc = df_he_euroSciVoc[df_he_euroSciVoc['projectID'].isin(filter_IT_he_projects)]
df_he_euroscivoc.shape

(9510, 5)

In [ ]:
# if df_he_euroscivoc comes from a filter → ensure it is a copy
df_he_euroscivoc = df_he_euroscivoc.copy()

# create column with the first level of the path
df_he_euroscivoc.loc[:, 'livello_1'] = (
    df_he_euroscivoc['euroSciVocPath']
        .astype(str)
        .str.split('/')
        .str[1]
)

# count
df_he_euroscivoc['livello_1'].value_counts().head(6)

livello_1
natural sciences               3646
engineering and technology     2260
social sciences                1501
medical and health sciences    1238
humanities                      436
agricultural sciences           429
Name: count, dtype: int64

In [43]:
# create a column with the second level of the euroscivoc path (if missing, the first level is inserted)
df_he_euroscivoc['livello_2'] = df_he_euroscivoc['euroSciVocPath'].str.split('/').str[2:3].str.join('/')

df_he_euroscivoc['livello_2'] = np.where(df_he_euroscivoc['livello_2'] == '', df_he_euroscivoc['livello_1'], df_he_euroscivoc['livello_2'])

print(df_he_euroscivoc["livello_2"].value_counts().head(5))

livello_2
biological sciences                                                        1010
computer and information sciences                                           848
physical sciences                                                           684
chemical sciences                                                           635
electrical engineering, electronic engineering, information engineering     605
Name: count, dtype: int64


In [ ]:
# filter the euroSciVocTitle and livello_1 for I2FVG companies (not to be used)
filter_i2fvg_he_euroscivoc = df_he_euroscivoc[df_he_euroscivoc['projectID'].isin(filter_i2fvg_he_project)]
print(f"The main euroSciVocTitle entries for I2FVG are: \n{filter_i2fvg_he_euroscivoc['euroSciVocTitle'].value_counts().head(5)}")
print("----------------------------------------")
print(f"The livello_1 entries for I2FVG are: \n{filter_i2fvg_he_euroscivoc['livello_1'].value_counts()}")
print("----------------------------------------")
print(f"The main livello_2 entries for I2FVG are: \n{filter_i2fvg_he_euroscivoc['livello_2'].value_counts().head(5)}")

I principali euroSciVocTitle i2FVG sono: 
euroSciVocTitle
energy and fuels    44
sensors             33
aircraft            26
governance          24
software            24
Name: count, dtype: int64
----------------------------------------
I livelli_1 i2FVG sono: 
livello_1
natural sciences               414
engineering and technology     341
social sciences                143
medical and health sciences     75
agricultural sciences           74
humanities                      16
Name: count, dtype: int64
----------------------------------------
I principali livello_2 i2FVG sono: 
livello_2
biological sciences                                                        115
computer and information sciences                                          110
environmental engineering                                                  108
chemical sciences                                                           79
electrical engineering, electronic engineering, information engineering     78
Name: c

## **Concatenate Dataframes**

In [45]:
df_organization_final = pd.concat(
    [df_h2020_organization, df_he_organization],
    ignore_index=True
)

df_project_final = pd.concat(
    [df_h2020_project, df_he_project],
    ignore_index=True
)

df_euroscivoc_final = pd.concat(
    [df_h2020_euroscivoc, df_he_euroscivoc],
    ignore_index=True
)

Final preparation of dataframes

In [46]:
# rename "id" column in "projectID"
df_project_final = df_project_final.rename(columns={
    'id':'projectID',
})

In [47]:
# replace "HORIZON" with "HORIZON EUROPE" in the 'frameworkProgramme' column
df_project_final['frameworkProgramme'] = df_project_final['frameworkProgramme'].str.replace('HORIZON', 'HORIZON EUROPE')

In [ ]:
# delete hours/minutes/seconds from datetime columns + change type object--->datetime64 (Using errors='coerce'. It will replace all non-numeric values with NaN)
df_organization_final["contentUpdateDate"] = pd.to_datetime(df_organization_final["contentUpdateDate"], errors="coerce", dayfirst=True).dt.date
df_organization_final["contentUpdateDate"] = pd.to_datetime(df_organization_final["contentUpdateDate"])

In [ ]:
# delete hours/minutes/seconds from datetime columns + change type object--->datetime64 (Using errors='coerce'. It will replace all non-numeric values with NaN)
list_date = ['startDate', 'endDate', 'ecSignatureDate', 'contentUpdateDate']

for col in list_date:
    df_project_final[col] = pd.to_datetime(df_project_final[col], errors='coerce')
    print(f"Column [{col}] fixed!")

Colonna [startDate] sistemata!
Colonna [endDate] sistemata!
Colonna [ecSignatureDate] sistemata!
Colonna [contentUpdateDate] sistemata!


## **Save to Output Folder**

In [50]:
#organization
csv_data = df_organization_final.to_csv(sep ='|', index=False).encode('utf-8-sig')
path = base_path / "data" / "eu_projects" / "merge" / "organization.csv"

with open(path, 'wb') as file:
    file.write(csv_data)


In [51]:
# project
csv_data = df_project_final.to_csv(sep ='|', index=False).encode('utf-8-sig')
path = base_path / "data" / "eu_projects" / "merge" / "project.csv"

with open(path, 'wb') as file:
    file.write(csv_data)

In [52]:
# euroscivoc
csv_data = df_euroscivoc_final.to_csv(sep ='|', index=False).encode('utf-8-sig')
path = base_path / "data" / "eu_projects" / "merge" / "euroscivoc.csv"
with open(path, 'wb') as file:
    file.write(csv_data)